In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense, Embedding, Input, Flatten, Conv1D, GlobalMaxPooling1D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

# 加载假新闻和真新闻数据
fake_df = pd.read_excel('E:/Fakenews Detection/Dataest_2.0/fake-and-real-news-dataset--Fake.xlsx', engine='openpyxl')
true_df = pd.read_excel('E:/Fakenews Detection/Dataest_2.0/fake-and-real-news-dataset--True.xlsx', engine='openpyxl')

# 添加标签
fake_df['label'] = 0
true_df['label'] = 1

# 合并数据
data_df = pd.concat([fake_df, true_df], ignore_index=True)
# 清理数据：删除或填充NaN值，确保所有文本数据都是字符串
data_df['text'].fillna('', inplace=True)
data_df['text'] = data_df['text'].astype(str)
# 提取文本和标签
X = data_df['text'].values
y = data_df['label'].values

# 分割数据集为训练和测试集
X_train_texts, X_test_texts, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 预处理文本数据
max_words = 10000
max_len = 150

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train_texts)
X_train_sequences = tokenizer.texts_to_sequences(X_train_texts)
X_test_sequences = tokenizer.texts_to_sequences(X_test_texts)

X_train = pad_sequences(X_train_sequences, maxlen=max_len)
X_test = pad_sequences(X_test_sequences, maxlen=max_len)

# 将标签转换为浮点数
y_train = y_train.astype(float)
y_test = y_test.astype(float)

# 加载 GloVe 词向量
embedding_dim = 300
embedding_file = 'E:\Fakenews Detection\code4.0\glove.6B\glove.6B.300d.txt'
embeddings_index = {}
with open(embedding_file, encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

embedding_matrix = np.zeros((max_words, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i < max_words:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

# CNN-Glove 模型
def create_cnn_glove_model():
    text_input = Input(shape=(max_len,), dtype=tf.int32, name='text_input')
    embedding_layer = Embedding(input_dim=max_words, output_dim=embedding_dim, weights=[embedding_matrix], input_length=max_len, trainable=False)(text_input)
    conv1 = Conv1D(filters=128, kernel_size=3, activation='relu')(embedding_layer)
    pool1 = GlobalMaxPooling1D()(conv1)
    conv2 = Conv1D(filters=64, kernel_size=4, activation='relu')(embedding_layer)
    pool2 = GlobalMaxPooling1D()(conv2)
    conv3 = Conv1D(filters=32, kernel_size=5, activation='relu')(embedding_layer)
    pool3 = GlobalMaxPooling1D()(conv3)
    concatenated = tf.concat([pool1, pool2, pool3], axis=-1)
    dropout = Dropout(0.5)(concatenated)
    output = Dense(1, activation='sigmoid')(dropout)
    model = Model(inputs=text_input, outputs=output)
    return model

# CNN-L2 正则化模型
def create_cnn_l2_regularized_model():
    text_input = Input(shape=(max_len,), dtype=tf.int32, name='text_input')
    embedding_layer = Embedding(input_dim=max_words, output_dim=128, input_length=max_len)(text_input)
    conv1 = Conv1D(filters=128, kernel_size=3, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(embedding_layer)
    pool1 = GlobalMaxPooling1D()(conv1)
    conv2 = Conv1D(filters=64, kernel_size=4, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(embedding_layer)
    pool2 = GlobalMaxPooling1D()(conv2)
    conv3 = Conv1D(filters=32, kernel_size=5, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(embedding_layer)
    pool3 = GlobalMaxPooling1D()(conv3)
    concatenated = tf.concat([pool1, pool2, pool3], axis=-1)
    dropout = Dropout(0.5)(concatenated)
    output = Dense(1, activation='sigmoid')(dropout)
    model = Model(inputs=text_input, outputs=output)
    return model

# 创建中间模型（对于这两种模型，可能不需要中间模型）
def create_middle_model(base_model):
    return None

# 生成对抗样本（简化生成随机扰动的对抗样本）
def generate_adversarial_samples(model, middle_model, X_sample, y_sample, epsilon=0.1):
    X_sample_tensor = tf.convert_to_tensor(X_sample, dtype=tf.float32) 
    y_sample_tensor = tf.convert_to_tensor(y_sample, dtype=tf.float32)
    # 生成随机扰动
    perturbations = tf.random.uniform(shape=X_sample_tensor.shape, minval=-epsilon, maxval=epsilon)
    X_adv_samples = X_sample_tensor + perturbations
    return X_adv_samples.numpy()

# 计算额外的评估指标
def evaluate_model(model, X, y_true):
    y_pred = (model.predict(X) > 0.5).astype(int)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return precision, recall, f1

# 聚合对抗性训练数据进行训练（通用函数，适用于两种模型）
def train_and_evaluate_adv(model, middle_model, X_train, y_train, X_test, y_test, epochs=10, epsilon=0.1):
    history_logs = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    precision_logs = {'train': [], 'val': []}
    recall_logs = {'train': [], 'val': []}
    f1_logs = {'train': [], 'val': []}

    model.compile(optimizer=Adam(learning_rate=1e-4), loss=BinaryCrossentropy(), metrics=['accuracy'])
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")

        # 生成对抗样本
        X_adv_samples = generate_adversarial_samples(model, middle_model, X_train, y_train, epsilon)

        # 合并对抗数据
        X_combined = np.concatenate((X_train, X_adv_samples), axis=0)
        y_combined = np.concatenate((y_train, y_train), axis=0)

        history = model.fit(X_combined, y_combined, epochs=1, batch_size=16, validation_split=0.2, verbose=1)

        for key in history_logs:
            history_logs[key].append(history.history[key][0])

        # 计算训练和验证集上的精确率、召回率和 F1 分数
        val_split_idx = len(X_combined) * 80 // 100
        precision_train, recall_train, f1_train = evaluate_model(model, X_combined[:val_split_idx], y_combined[:val_split_idx])
        precision_val, recall_val, f1_val = evaluate_model(model, X_combined[val_split_idx:], y_combined[val_split_idx:])

        precision_logs['train'].append(precision_train)
        recall_logs['train'].append(recall_train)
        f1_logs['train'].append(f1_train)
        precision_logs['val'].append(precision_val)
        recall_logs['val'].append(recall_val)
        f1_logs['val'].append(f1_val)

    # 在原始测试集上进行评估（不结合对抗样本）
    _, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f'对抗训练模型在原始测试集上的准确率: {accuracy:.4f}')

    precision, recall, f1 = evaluate_model(model, X_test, y_test)
    print(f'对抗训练模型在原始测试集上的精确率: {precision:.4f}, 召回率: {recall:.4f}, F1 分数: {f1:.4f}')

    # 生成并评估对抗测试数据
    X_test_adv_samples = generate_adversarial_samples(model, middle_model, X_test, y_test, epsilon)

    _, accuracy_adv = model.evaluate(X_test_adv_samples, y_test, verbose=0)
    print(f'对抗训练模型在对抗测试集上的准确率: {accuracy_adv:.4f}')

    precision_adv, recall_adv, f1_adv = evaluate_model(model, X_test_adv_samples, y_test)
    print(f'对抗训练模型在对抗测试集上的精确率: {precision_adv:.4f}, 召回率: {recall_adv:.4f}, F1 分数: {f1_adv:.4f}')

    return model, history_logs, precision_logs, recall_logs, f1_logs

# 创建并训练 CNN-Glove 模型
cnn_glove_model = create_cnn_glove_model()
print("\n训练 CNN-Glove 模型...")
cnn_glove_model, cnn_glove_history, cnn_glove_precision_logs, cnn_glove_recall_logs, cnn_glove_f1_logs = train_and_evaluate_adv(
    cnn_glove_model, None, X_train, y_train, X_test, y_test, epochs=10)

# 创建并训练 CNN-L2 正则化模型
cnn_l2_model = create_cnn_l2_regularized_model()
print("\n训练 CNN-L2 正则化模型...")
cnn_l2_model, cnn_l2_history, cnn_l2_precision_logs, cnn_l2_recall_logs, cnn_l2_f1_logs = train_and_evaluate_adv(
    cnn_l2_model, None, X_train, y_train, X_test, y_test, epochs=10)


训练 CNN-Glove 模型...
Epoch 1/10
3592/3592 [==============================] - 204s 57ms/step - loss: 0.2996 - accuracy: 0.8675 - val_loss: 0.2553 - val_accuracy: 0.9053
Epoch 2/10
3592/3592 [==============================] - 201s 56ms/step - loss: 0.1570 - accuracy: 0.9411 - val_loss: 0.2022 - val_accuracy: 0.9247
Epoch 3/10
3592/3592 [==============================] - 201s 56ms/step - loss: 0.1274 - accuracy: 0.9540 - val_loss: 0.1690 - val_accuracy: 0.9406
Epoch 4/10
3592/3592 [==============================] - 204s 57ms/step - loss: 0.1067 - accuracy: 0.9630 - val_loss: 0.1472 - val_accuracy: 0.9483
Epoch 5/10
3592/3592 [==============================] - 204s 57ms/step - loss: 0.0915 - accuracy: 0.9688 - val_loss: 0.1312 - val_accuracy: 0.9548
Epoch 6/10
3592/3592 [==============================] - 204s 57ms/step - loss: 0.0812 - accuracy: 0.9722 - val_loss: 0.1195 - val_accuracy: 0.9574
Epoch 7/10
3592/3592 [==============================] - 198s 55ms/step - loss: 0.0740 - accuracy: 